# Digital Banking Transaction Forecasting & Anomaly Analytics

End-to-end time-series forecasting and anomaly analytics on a **synthetic** digital-banking transaction dataset. The notebook aggregates transaction events to daily KPIs, creates leakage-safe lag features, compares forecasting models, and detects unusual days with Isolation Forest.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
df = pd.read_csv('../data/raw/digital_banking_transactions.csv', parse_dates=['transaction_timestamp'])
print(df.shape)
df.head()


## 1. Data quality checks


In [ ]:
df.info()
print(df.isna().sum())
print('Duplicates:', df.duplicated().sum())


## 2. Daily KPI aggregation


In [ ]:
df['transaction_date'] = df['transaction_timestamp'].dt.floor('D')
daily = df.groupby('transaction_date').agg(
    transaction_count=('transaction_id','count'),
    total_transaction_value_inr=('transaction_amount_inr','sum'),
    avg_transaction_value_inr=('transaction_amount_inr','mean'),
    failed_transactions=('status', lambda s: (s == 'Failed').sum()),
    unique_customers=('customer_id','nunique')
).reset_index()
daily['failure_rate_pct'] = 100 * daily['failed_transactions'] / daily['transaction_count']
daily.head()


## 3. Trend and seasonality


In [ ]:
plt.figure(figsize=(14,5))
plt.plot(daily['transaction_date'], daily['total_transaction_value_inr'])
plt.title('Daily Transaction Value')
plt.xlabel('Date'); plt.ylabel('INR'); plt.show()


## 4. Leakage-safe forecasting features
All rolling statistics are shifted by one day so the model never uses the target day to predict itself.


In [ ]:
f = daily[['transaction_date','total_transaction_value_inr']].copy()
f['day_of_week'] = f['transaction_date'].dt.dayofweek
f['month'] = f['transaction_date'].dt.month
f['day_of_month'] = f['transaction_date'].dt.day
for lag in [1,7,14,28]:
    f[f'lag_{lag}'] = f['total_transaction_value_inr'].shift(lag)
f['rolling_mean_7'] = f['total_transaction_value_inr'].shift(1).rolling(7).mean()
f['rolling_mean_28'] = f['total_transaction_value_inr'].shift(1).rolling(28).mean()
f['rolling_std_28'] = f['total_transaction_value_inr'].shift(1).rolling(28).std()
f = f.dropna().reset_index(drop=True)


## 5. Chronological train/test split and model comparison


In [ ]:
features = [c for c in f.columns if c not in ['transaction_date','total_transaction_value_inr']]
cut = int(len(f) * 0.8)
train, test = f.iloc[:cut], f.iloc[cut:]
X_train, y_train = train[features], train['total_transaction_value_inr']
X_test, y_test = test[features], test['total_transaction_value_inr']

models = {
    'Random Forest': RandomForestRegressor(n_estimators=220, min_samples_leaf=3, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=180, learning_rate=0.04, max_depth=3, random_state=42)
}
results = []
baseline = X_test['lag_7'].to_numpy()
results.append(['Seasonal Naive (Lag 7)', mean_absolute_error(y_test, baseline), np.sqrt(mean_squared_error(y_test, baseline)), r2_score(y_test, baseline)])
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append([name, mean_absolute_error(y_test,pred), np.sqrt(mean_squared_error(y_test,pred)), r2_score(y_test,pred)])
pd.DataFrame(results, columns=['model','MAE','RMSE','R2']).sort_values('RMSE')


## 6. Anomaly detection with Isolation Forest


In [ ]:
anomaly_features = ['transaction_count','total_transaction_value_inr','avg_transaction_value_inr','failure_rate_pct']
iso = IsolationForest(n_estimators=250, contamination=0.025, random_state=42)
daily['anomaly_flag'] = (iso.fit_predict(daily[anomaly_features]) == -1).astype(int)
daily['anomaly_score'] = -iso.decision_function(daily[anomaly_features])
daily[daily['anomaly_flag'] == 1].sort_values('anomaly_score', ascending=False).head(10)


## 7. Business interpretation
Flagged days are investigation candidates, not proof of fraud. Forecast errors and anomalies can support operations planning, monitoring, and root-cause analysis.
